# Clase 077 — SVM lineal

Las **Support Vector Machines** lineales buscan el hiperplano que **maximiza el margen**
entre las clases. En esta clase entrenamos un `LinearSVC` sobre **Iris** (Virginica vs. resto),
vemos por qué el **escalado** es crítico, y estudiamos cómo el hiperparámetro **C** regula el
trade-off entre un margen ancho y las violaciones permitidas (*soft margin*).

Requiere: `numpy`, `matplotlib`, `scikit-learn`.

## 1. Datos: Iris binarizado (Virginica vs. resto)

Nos quedamos con dos features (*petal length*, *petal width*) para poder visualizar la
frontera en 2D, y convertimos el problema a binario.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

RND = 42
np.random.seed(RND)
iris = load_iris()
X = iris.data[:, 2:4]            # petalo: largo y ancho
y = (iris.target == 2).astype(int)   # 1 = Virginica, 0 = resto
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RND, stratify=y)
print("X:", X.shape, "| Virginica (positivos):", int(y.sum()))
print("train:", X_train.shape[0], "| test:", X_test.shape[0])

## 2. Pipeline StandardScaler + LinearSVC

SVM es sensible a la escala: sin `StandardScaler` la variable de mayor rango domina el margen.
Entrenamos con `C=1` y `loss="hinge"` (la pérdida SVM canónica del libro).

In [ ]:
clf = make_pipeline(
    StandardScaler(),
    LinearSVC(C=1, loss="hinge", dual=True, max_iter=5000, random_state=RND))
clf.fit(X_train, y_train)

acc_train = accuracy_score(y_train, clf.predict(X_train))
acc_test = accuracy_score(y_test, clf.predict(X_test))
print(f"accuracy train (C=1): {acc_train:.3f}")
print(f"accuracy test  (C=1): {acc_test:.3f}")
assert acc_test >= 0.9, "el SVM lineal escalado debe rendir bien en Iris"
print("OK: separacion lineal casi perfecta sobre los petalos")

## 3. Efecto del hiperparámetro C

**C alto** → poco tolerante, margen estrecho, riesgo de *overfitting*.
**C bajo** → más tolerante, margen ancho, más regularización.
Ojo: es **inverso** al `alpha` de Ridge/Lasso.

In [ ]:
print(f"{'C':>7} | {'acc_train':>9} | {'acc_test':>8}")
for C in [0.1, 1, 100]:
    m = make_pipeline(
        StandardScaler(),
        LinearSVC(C=C, loss="hinge", dual=True, max_iter=5000, random_state=RND))
    m.fit(X_train, y_train)
    at = accuracy_score(y_train, m.predict(X_train))
    av = accuracy_score(y_test, m.predict(X_test))
    print(f"{C:>7} | {at:>9.3f} | {av:>8.3f}")

## 4. Frontera de decisión y margen

Entrenamos `SVC(kernel="linear")` sobre los datos ya escalados para poder marcar los
**vectores soporte** (`support_vectors_`, que `LinearSVC` no expone) y dibujar el margen
con las curvas de `decision_function` en -1, 0 y +1.

In [ ]:
scaler = StandardScaler().fit(X_train)
Xs = scaler.transform(X_train)
svc = SVC(kernel="linear", C=1).fit(Xs, y_train)
print("vectores soporte:", svc.support_vectors_.shape[0])

x0 = np.linspace(Xs[:, 0].min() - 0.5, Xs[:, 0].max() + 0.5, 200)
x1 = np.linspace(Xs[:, 1].min() - 0.5, Xs[:, 1].max() + 0.5, 200)
xx, yy = np.meshgrid(x0, x1)
Z = svc.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contour(xx, yy, Z, levels=[-1, 0, 1],
           colors="k", linestyles=["--", "-", "--"])
ax.scatter(Xs[:, 0], Xs[:, 1], c=y_train, cmap="coolwarm", edgecolor="k", s=30)
ax.scatter(svc.support_vectors_[:, 0], svc.support_vectors_[:, 1],
           s=140, facecolors="none", edgecolors="lime", linewidths=2,
           label="vectores soporte")
ax.set_xlabel("petal length (escalado)"); ax.set_ylabel("petal width (escalado)")
ax.set_title("SVM lineal: frontera (solida) y margen (punteado)")
ax.legend(); plt.tight_layout(); plt.show()

## 5. Robustez frente a un outlier

Agregamos un outlier artificial de la clase Virginica en zona ajena y comparamos `C` alto vs.
`C` bajo. Un `C` bajo (más regularización) absorbe mejor el punto anómalo.

In [ ]:
X_out = np.vstack([Xs, [[-1.5, 1.5]]])   # outlier en zona de la clase 0
y_out = np.append(y_train, 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, C in zip(axes, [0.05, 100]):
    m = SVC(kernel="linear", C=C).fit(X_out, y_out)
    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_out[:, 0], X_out[:, 1], c=y_out, cmap="coolwarm",
               edgecolor="k", s=30)
    ax.scatter([-1.5], [1.5], marker="X", s=200, c="yellow",
               edgecolor="k", label="outlier")
    ax.set_title(f"C = {C}"); ax.legend()
plt.suptitle("C bajo (izq) es mas robusto al outlier que C alto (der)")
plt.tight_layout(); plt.show()
print("C alto se deforma para clasificar el outlier; C bajo lo ignora")

## Ejercicios

1. Entrená el `Pipeline([StandardScaler, LinearSVC(C=1, loss="hinge")])` y reportá `accuracy`
   sobre un split train/test estratificado (ejercicio 1 del README).
2. Repetí con `C ∈ {0.1, 1, 100}` y compará *accuracy* de train y test. ¿Qué ocurre en los
   extremos del rango de `C`?
3. Graficá la frontera de decisión y el margen (líneas punteadas) para `C=1`, marcando los
   vectores soporte.
4. Agregá un outlier a la clase minoritaria y volvé a entrenar con `C` alto y `C` bajo.
   Mostrá visualmente cómo `C` bajo absorbe mejor el outlier.

## Conclusiones

- SVM lineal maximiza el **margen**; solo los **vectores soporte** definen la frontera.
- El **escalado** (`StandardScaler`) es obligatorio: sin él, el margen lo domina la feature
  de mayor rango.
- `C` regula el *soft margin*: **alto** = margen estrecho y menos regularización; **bajo** =
  margen ancho y más regularización (inverso a `alpha` de Ridge/Lasso).
- `LinearSVC` no expone `support_vectors_` ni `predict_proba`; para inspeccionarlos usá
  `SVC(kernel="linear")` o `CalibratedClassifierCV`.